# HDB resale price classification — EDA
Run from the project root. Shared cleaning validates and encodes the target; display-name normalization below is exploratory only. Training uses categorical IDs. Use `python main.py` for training and held-out evaluation.


In [ ]:
import numpy as np
import pandas as pd
from src.data_preparation import load_data, parse_remaining_lease, split_data

df = load_data("data/data.csv")
print(f"Shape after basic cleaning: {df.shape}")
df.head()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
# Set the aesthetic style of the plots
sns.set_style("whitegrid")

# Bar chart for price category with annotations
plt.figure(figsize=(10, 6))
count_plot = sns.countplot(x='price_category', hue='price_category', data=df, palette='pastel', legend=False)
plt.title('Count of Flats in Each Price Category (Target)')
plt.xlabel('Price Category')
plt.ylabel('Count')
#plt.xticks(rotation=45)

# Annotate the bars with the frequency count
for p in count_plot.patches:
    count_plot.annotate(format(p.get_height(), '.0f'),
                        (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha = 'center', va = 'center',
                        xytext = (0, 9),
                        textcoords = 'offset points')

plt.show()

In [ ]:
# Target labels are validated and encoded by load_data.
df['price_category'].value_counts()


In [ ]:
# 1. Inspect unique values and counts for flat_type specifically
print("--- Unique values in flat_type ---")
print(sorted(df["flat_type"].dropna().unique()))

print("\n--- Frequency count of flat_type ---")
print(df["flat_type"].value_counts(dropna=False))

print("\n" + "=" * 50 + "\n")

# 2. Inspect unique value counts across all categorical/text columns
text_cols = df.select_dtypes(include=["object", "string"]).columns

for col in text_cols:
    print(f"=== Column: {col} | Unique count: {df[col].nunique()} ===")
    print(df[col].value_counts(dropna=False))
    print("-" * 40)

In [ ]:
# Normalize flat type labels.
df['flat_type'] = df['flat_type'].replace({'FOUR ROOM': '4 ROOM'})

print("--- Updated flat_type counts ---")
print(df['flat_type'].value_counts(dropna=False))

In [ ]:
# Preview lease conversion without modifying the raw schema.
df["remaining_lease"].map(parse_remaining_lease).head(10)


In [ ]:
df.head()

In [ ]:
# Explore missing display names; model features use categorical IDs.
df[["town_name", "flatm_name"]].isna().sum()


In [ ]:
# Normalize model display names for exploratory analysis only.
import numpy as np

target_col = "flatm_name"

df[target_col] = df[target_col].astype(str).str.strip().str.upper()

model_map = {
    "PREMIUM APARTMENT.": "PREMIUM APARTMENT",
    "MULTI GENERATION": "MULTI-GENERATION",
    "NAN": np.nan,
    "NONE": np.nan,
    "2-ROOM": np.nan,
}

df[target_col] = df[target_col].replace(model_map)

print(df[target_col].value_counts(dropna=False))

In [ ]:
import yaml

with open("src/config.yaml", encoding="utf-8") as stream:
    config = yaml.safe_load(stream)
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df, config["split"])
print({"train": len(y_train), "validation": len(y_val), "test": len(y_test)})
